# Workshop 3 · 02 — Kommentare: Sentiment & Keywords

Aus den `kunden_kommentar`-Freitexten gewinnen wir **Sentiment** und **Themen/Keywords** — wieder eine Zeile pro Transformation, ohne NLP-Modell zu trainieren.

- `ai.analyze_sentiment` — Stimmung mit eigenen Labels
- `ai.extract` — Themen & konkrete Probleme als Entitäten

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.spark.aifunc as aifunc

base = spark.table("asset_clean")

sent = base.ai.analyze_sentiment(
    input_col="kunden_kommentar",
    output_col="sentiment",
    labels=["begeistert", "zufrieden", "neutral", "veraergert"],
)
display(sent.select("meldung_id", "kunden_kommentar", "sentiment"))

## Keywords & Themen mit `ai.extract`

Statt Stoppwortlisten und Regex: `ai.extract` zieht **Thema** und **konkretes Problem** als Entitäten direkt aus dem Kommentar.

In [ ]:
# This code uses AI. Always review output for mistakes.
from pyspark.sql.functions import col

kw = sent.ai.extract(
    labels=["thema", "konkretes_problem"],
    input_col="kunden_kommentar",
)

enriched = kw.select("meldung_id", "depot_norm", "baureihe_norm",
                     "kunden_kommentar", "sentiment", "thema", "konkretes_problem")
enriched.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("kommentar_insights")
display(enriched)

In [ ]:
from pyspark.sql.functions import count

# Aggregat: Stimmung je Depot -> direkt Power-BI-fähig (Direct Lake)
matrix = (spark.table("kommentar_insights")
          .groupBy("depot_norm", "sentiment").agg(count("*").alias("n"))
          .orderBy("depot_norm", "sentiment"))
display(matrix)

### Takeaway

Freitext-Kommentare → **Sentiment + Themen** als Tabelle `kommentar_insights`. Die Aggregat-Matrix ist direkt ein **Power-BI-Visual** (Direct Lake) — ohne Datenexport. Weitere nützliche Funktionen: `ai.summarize` (Management-Summary), `ai.translate` (mehrsprachiges Feedback).